# Retrieval & Explanation Faithfulness on SciFact — single-notebook edition

**Paper:** *Do Retrieved Evidence Improve Explanation Faithfulness? An Empirical Study on Biomedical Claim Verification.*

This one notebook runs the whole pipeline: data prep → BM25/Dense/Hybrid retrieval → LLM verification (no-retrieval vs retrieval) → faithfulness metrics → error analysis → figures.

### Before you run (Kaggle setup)
1. **Add your dataset:** right sidebar → *Add Input* → your private SciFact dataset (with `corpus.jsonl`, `claims_train.jsonl`, `claims_dev.jsonl`). The notebook auto-detects the path.
2. **Turn on the GPU:** right sidebar → *Session options* → *Accelerator* → **GPU T4 x2** (or P100).
3. **Turn on Internet:** *Session options* → **Internet: On** (needed to download the HuggingFace models the first time).
4. Run all cells (`Run → Run all`).

Set `QUICK_TEST = True` in the config cell for a fast sanity pass on a handful of claims before the full run.

All outputs are written to `/kaggle/working/` (metrics JSON, prediction JSONL, an error-analysis CSV, and PDF/PNG figures) so you can download them from the *Output* tab.

## 1. Install dependencies

In [1]:
# torch, scikit-learn, matplotlib, pandas are preinstalled on Kaggle.
# Install the retrieval / LLM extras. (~1-2 min the first time.)
!pip -q install rank-bm25 faiss-cpu "sentence-transformers>=2.6.0" "transformers>=4.40" accelerate 2>/dev/null
print("dependencies ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 75.6 MB/s eta 0:00:00
dependencies ready


## 2. Configuration & dataset auto-detection

Edit the flags here. The dataset path is found automatically by searching
`/kaggle/input` for `corpus.jsonl`.

In [2]:
import os, glob, json, random, re, platform
import importlib.metadata as md
from pathlib import Path
import numpy as np

# ----------------------------- knobs -------------------------------------- #
QUICK_TEST      = False   # True -> only run on LIMIT claims (fast smoke test)
LIMIT           = 25      # used when QUICK_TEST is True
USE_NLI         = True    # NLI-based faithfulness (False = lexical overlap only)
RETRIEVER_FOR_LLM = "hybrid"   # which retriever feeds the LLM: bm25 | dense | hybrid

# models (all downloaded from HuggingFace the first time; needs Internet ON)
DENSE_ENCODER = "pritamdeka/S-PubMedBert-MS-MARCO"          # biomedical encoder
LLM_MODEL     = os.environ.get("LLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")   # rerun with a 7B/8B model
LLM_TAG       = LLM_MODEL.split("/")[-1]
NLI_MODEL     = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# experiment settings
TOP_K_VALUES        = [1, 3, 5]
RETRIEVAL_K_FOR_LLM = 5
HYBRID_ALPHA        = 0.5
LABELS              = ["SUPPORT", "CONTRADICT", "NEI"]
FAITH_ENTAIL_THRESH = 0.5
LEXICAL_THRESH      = 0.3
SEED                = 42
MAX_NEW_TOKENS      = 400
BATCH_SIZE          = 32
N_BOOT              = 5000   # ONE value everywhere (paper currently says 1,000)
HAL_TH              = 0.5    # "faithful" cutoff for the error analysis
LEX_GRID            = [0.2, 0.3, 0.4, 0.5]
ENT_GRID            = [0.5, 0.7, 0.9]

# ----------------------------- paths -------------------------------------- #
_BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
OUT = _BASE / LLM_TAG            # one folder per verifier (do NOT redefine OUT below)
OUT.mkdir(parents=True, exist_ok=True)

def _find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        hits = sorted(glob.glob(f"./**/{name}", recursive=True))
    return hits[0] if hits else None

CORPUS_FILE = _find("corpus.jsonl")
TRAIN_FILE  = _find("claims_train.jsonl")
EVAL_FILE   = _find("claims_dev.jsonl")   # dev is the eval split (test has no labels)

assert CORPUS_FILE and EVAL_FILE, (
    "Could not find corpus.jsonl / claims_dev.jsonl under /kaggle/input. "
    "Attach your SciFact dataset via 'Add Input'."
)
print("corpus:", CORPUS_FILE)
print("train :", TRAIN_FILE)
print("eval  :", EVAL_FILE)
print("output:", OUT)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass
set_seed(SEED)

import torch
print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ----------------------------- versions / config -------------------------- #
def _ver(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return "n/a"

VERSIONS = {p: _ver(p) for p in ["torch", "transformers", "sentence-transformers", "rank-bm25",
                                 "faiss-cpu", "scikit-learn", "scipy", "numpy", "pandas"]}
CONFIG = dict(LLM_MODEL=LLM_MODEL, DENSE_ENCODER=DENSE_ENCODER, NLI_MODEL=NLI_MODEL,
              SEED=SEED, HYBRID_ALPHA=HYBRID_ALPHA, K_FOR_LLM=RETRIEVAL_K_FOR_LLM,
              LEXICAL_THRESH=LEXICAL_THRESH, FAITH_ENTAIL_THRESH=FAITH_ENTAIL_THRESH,
              MAX_NEW_TOKENS=MAX_NEW_TOKENS, DECODING="greedy (do_sample=False)",
              BM25="rank_bm25.BM25Okapi defaults (k1=1.5,b=0.75,eps=0.25); lowercase [a-z0-9]+; no stemming/stopwords",
              HYBRID_POOL=50, N_BOOT=N_BOOT, python=platform.python_version(), versions=VERSIONS)
# NOTE: write_json is defined in cell 6 -> config is saved at the end of cell 6.

corpus: /kaggle/input/datasets/abrarhafiz/scifact-dataset/corpus.jsonl
train : /kaggle/input/datasets/abrarhafiz/scifact-dataset/claims_train.jsonl
eval  : /kaggle/input/datasets/abrarhafiz/scifact-dataset/claims_dev.jsonl
output: /kaggle/working/Qwen2.5-3B-Instruct
CUDA available: True | device: Tesla T4


## 3. Helper functions

In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2)

_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")
_WORD_RE = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    text = (text or "").strip()
    if not text: return []
    out = []
    for para in re.split(r"[\n\r]+", text):
        para = re.sub(r"^\s*[-*\u2022\d.)]+\s*", "", para).strip()
        if not para: continue
        out.extend(s.strip() for s in _SENT_SPLIT_RE.split(para) if s.strip())
    return out

def tokenize(text):
    return _WORD_RE.findall((text or "").lower())

def jaccard(a, b):
    ta, tb = set(tokenize(a)), set(tokenize(b))
    if not ta or not tb: return 0.0
    return len(ta & tb) / len(ta | tb)

def minmax(x):
    x = np.asarray(x, dtype=np.float64)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-12: return np.zeros_like(x)
    return (x - lo) / (hi - lo)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(o):
        run = max(run, (m - rank) * p[i]); adj[i] = min(run, 1.0)
    return adj
def nanmean(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return float(x.mean()) if len(x) else float("nan")

write_json(OUT / "config.json", CONFIG)

## 4. Data preparation

Builds the flat retrieval corpus and the labeled eval examples. Claim label is
derived from the SciFact `evidence` dict: any SUPPORT → SUPPORT, else any
CONTRADICT → CONTRADICT, empty → NEI. `gold_doc_ids` comes from `cited_doc_ids`.

In [4]:
def build_corpus(path):
    docs = []
    for row in read_jsonl(path):
        doc_id = str(row["doc_id"])
        title = (row.get("title") or "").strip()
        sents = [s.strip() for s in row.get("abstract", []) if s and s.strip()]
        text = (title + ". " + " ".join(sents)).strip()
        docs.append({"doc_id": doc_id, "title": title, "abstract": sents,
                     "sentences": sents, "text": text})
    return docs

def _claim_label(evidence):
    labels = [it["label"].upper() for ev in evidence.values() for it in ev]
    if any(l == "SUPPORT" for l in labels): return "SUPPORT"
    if any(l == "CONTRADICT" for l in labels): return "CONTRADICT"
    return "NEI"

def build_examples(path):
    if not path: return []
    ex = []
    for row in read_jsonl(path):
        evidence = row.get("evidence", {}) or {}
        gold = [str(d) for d in row.get("cited_doc_ids", [])] or \
               [str(d) for d in evidence.keys()]
        gold_ev = {str(k): sorted({s for it in v for s in it.get("sentences", [])})
                   for k, v in evidence.items()}
        ex.append({"claim_id": row["id"], "claim": row["claim"].strip(),
                   "label": _claim_label(evidence),
                   "gold_doc_ids": gold, "gold_evidence": gold_ev})
    return ex

corpus = build_corpus(CORPUS_FILE)
corpus_by_id = {d["doc_id"]: d for d in corpus}
eval_examples = build_examples(EVAL_FILE)
if QUICK_TEST:
    eval_examples = eval_examples[:LIMIT]

write_jsonl(OUT / "corpus.jsonl", corpus)
write_jsonl(OUT / "eval_examples.jsonl", eval_examples)

from collections import Counter
print(f"corpus docs      : {len(corpus):,}")
print(f"eval claims      : {len(eval_examples):,}")
print("label distribution:", dict(Counter(e['label'] for e in eval_examples)))
n_nei_with_doc = sum(1 for e in eval_examples if e["label"]=="NEI" and e["gold_doc_ids"])
print("NEI claims that still have cited_doc_ids:", n_nei_with_doc, "/",
      sum(e["label"]=="NEI" for e in eval_examples))

corpus docs      : 5,183
eval claims      : 300
label distribution: {'NEI': 112, 'SUPPORT': 124, 'CONTRADICT': 64}
NEI claims that still have cited_doc_ids: 112 / 112


## 5. Dense index

Embeds every abstract with the biomedical encoder. Uses FAISS if available,
otherwise falls back to an exact numpy cosine search (fine for a few thousand
docs).

In [5]:
from sentence_transformers import SentenceTransformer

print(f"Loading encoder: {DENSE_ENCODER}")
_encoder = SentenceTransformer(DENSE_ENCODER)

_texts = [d["text"] for d in corpus]
_doc_ids = [d["doc_id"] for d in corpus]
print(f"Encoding {len(_texts):,} documents ...")
_emb = _encoder.encode(_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype("float32")

class DenseIndex:
    def __init__(self, emb, doc_ids, encoder):
        self.doc_ids = doc_ids; self.encoder = encoder
        try:
            import faiss
            self.index = faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb)
            self.backend = "faiss"
        except Exception as e:
            print("faiss unavailable, using numpy fallback:", e)
            self.emb = emb; self.index = None; self.backend = "numpy"
    def search(self, query, k):
        q = self.encoder.encode([query], convert_to_numpy=True,
                                normalize_embeddings=True).astype("float32")
        if self.backend == "faiss":
            scores, idxs = self.index.search(q, k)
            return [(self.doc_ids[i], float(s))
                    for s, i in zip(scores[0], idxs[0]) if i != -1]
        sims = (self.emb @ q[0])
        top = np.argsort(-sims)[:k]
        return [(self.doc_ids[i], float(sims[i])) for i in top]

dense_index = DenseIndex(_emb, _doc_ids, _encoder)
print("dense index ready via", dense_index.backend)

Loading encoder: pritamdeka/S-PubMedBert-MS-MARCO


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

... (8 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 5,183 documents ...


Batches:   0%|          | 0/162 [00:00<?, ?it/s]

dense index ready via faiss


## 6. Retrievers — BM25 / Dense / Hybrid

In [6]:
from rank_bm25 import BM25Okapi

class BM25Retriever:
    name = "bm25"
    def __init__(self, docs):
        self.docs = docs; self.doc_ids = [d["doc_id"] for d in docs]
        self.bm25 = BM25Okapi([tokenize(d["text"]) for d in docs])
    def scores(self, q): return np.asarray(self.bm25.get_scores(tokenize(q)), dtype=np.float64)
    def retrieve(self, q, k):
        s = self.scores(q); top = np.argsort(-s)[:k]
        return [(self.doc_ids[i], float(s[i])) for i in top]

class DenseRetriever:
    name = "dense"
    def __init__(self, index): self.index = index
    def retrieve(self, q, k): return self.index.search(q, k)

class HybridRetriever:
    name = "hybrid"
    def __init__(self, bm25, dense, alpha=HYBRID_ALPHA, pool=50):
        self.bm25, self.dense, self.alpha, self.pool = bm25, dense, alpha, pool
    def retrieve(self, q, k):
        pool = max(self.pool, k)
        b_all = self.bm25.scores(q)
        b_idx = np.argsort(-b_all)[:pool]
        b_cand = {self.bm25.doc_ids[i]: float(b_all[i]) for i in b_idx}
        d_cand = dict(self.dense.retrieve(q, pool))
        ids = list(set(b_cand) | set(d_cand))
        bmin = min(b_cand.values(), default=0.0); dmin = min(d_cand.values(), default=0.0)
        bv = minmax([b_cand.get(i, bmin) for i in ids])
        dv = minmax([d_cand.get(i, dmin) for i in ids])
        fused = self.alpha * dv + (1 - self.alpha) * bv
        order = np.argsort(-fused)[:k]
        return [(ids[i], float(fused[i])) for i in order]

bm25   = BM25Retriever(corpus)
dense  = DenseRetriever(dense_index)
hybrid = HybridRetriever(bm25, dense)
RETRIEVERS = {"bm25": bm25, "dense": dense, "hybrid": hybrid}
print("retrievers ready:", list(RETRIEVERS))

retrievers ready: ['bm25', 'dense', 'hybrid']


## 7. Retrieval evaluation — Experiment 1

Recall@1/3/5 and MRR against gold cited abstracts, and caches the retrieved
doc_ids per claim for the LLM stage.

In [7]:
from tqdm.auto import tqdm

def recall_at_k(retrieved, gold, k):
    if not gold: return float("nan")
    return len(set(retrieved[:k]) & gold) / len(gold)

def reciprocal_rank(retrieved, gold):
    for r, d in enumerate(retrieved, 1):
        if d in gold: return 1.0 / r
    return 0.0

max_k = max(TOP_K_VALUES + [RETRIEVAL_K_FOR_LLM])
retrieval_metrics = {}
retrieved_cache = {}   # method -> {claim_id: [doc_id,...]}

for name, retr in RETRIEVERS.items():
    per_k = {k: [] for k in TOP_K_VALUES}; rr = []; cache = {}
    rows_pc = []                                   # per-claim scores, saved for paired tests

    for ex in tqdm(eval_examples, desc=f"[{name}] retrieving"):
        gold = set(ex["gold_doc_ids"])
        docs = [d for d, _ in retr.retrieve(ex["claim"], max_k)]
        cache[ex["claim_id"]] = docs
        if not gold: continue
        rows_pc.append({"claim_id": ex["claim_id"],
                        **{f"recall@{k}": recall_at_k(docs, gold, k) for k in TOP_K_VALUES},
                        "rr": reciprocal_rank(docs, gold)})
        for k in TOP_K_VALUES: per_k[k].append(recall_at_k(docs, gold, k))
        rr.append(reciprocal_rank(docs, gold))

    retrieved_cache[name] = cache
    m = {f"recall@{k}": float(np.mean(v)) for k, v in per_k.items() if v}
    m["mrr"] = float(np.mean(rr)) if rr else 0.0
    m["num_scored"] = len(rr)
    retrieval_metrics[name] = m

    write_jsonl(OUT / f"retrieval_perclaim_{name}.jsonl", rows_pc)
    write_jsonl(OUT / f"retrieved_{name}.jsonl",
                [{"claim_id": c, "doc_ids": d} for c, d in cache.items()])

write_json(OUT / "retrieval_metrics.json", retrieval_metrics)
import pandas as pd
display(pd.DataFrame(retrieval_metrics).T)

[bm25] retrieving:   0%|          | 0/300 [00:00<?, ?it/s]

[dense] retrieving:   0%|          | 0/300 [00:00<?, ?it/s]

[hybrid] retrieving:   0%|          | 0/300 [00:00<?, ?it/s]

,recall@1,recall@3,recall@5,mrr,num_scored
bm25,0.507500,0.673333,0.726778,0.612556,300.0
dense,0.530111,0.684944,0.751722,0.638167,300.0
hybrid,0.574000,0.716889,0.760500,0.671333,300.0


## 8. LLM verification — Setting A vs B

Loads the verifier LLM once and generates a label + explanation for every claim,
with and without retrieved evidence. Greedy decoding for reproducibility.

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter

SYSTEM_VERIFY = ("You are a careful biomedical fact-checker. You decide whether a "
    "scientific claim is Supported, Refuted, or has Not Enough Information, and you "
    "explain your reasoning grounded ONLY in the text you are given.")

VERIFY_WITH_EVIDENCE = '''Claim:
{claim}

Retrieved Evidence:
{evidence}

Task:
1. In one sentence, summarize what the retrieved evidence is actually about.
2. State the label. Exactly one of: SUPPORT, CONTRADICT, NEI.
   - SUPPORT: the evidence supports the claim.
   - CONTRADICT: the evidence refutes the claim.
   - NEI: the evidence does not contain enough information to decide.
3. Explain your verdict in 1-3 sentences, using ONLY information in the evidence:
   - If SUPPORT or CONTRADICT, cite the specific part of the evidence that
     justifies it.
   - If NEI, first say what the evidence DOES establish, then explain why that
     is insufficient to decide the claim.

Respond with ONLY a JSON object on one line, no markdown:
{{"label": "<SUPPORT|CONTRADICT|NEI>", "evidence_summary": "<one sentence>", "explanation": "<your explanation>"}}'''

VERIFY_NO_EVIDENCE = '''Claim:
{claim}

Task:
1. Decide the label using your own knowledge. Exactly one of: SUPPORT, CONTRADICT, NEI.
2. Explain WHY in 1-4 sentences.

Respond with ONLY a JSON object on one line, no markdown:
{{"label": "<SUPPORT|CONTRADICT|NEI>", "explanation": "<your explanation>"}}'''

# A2: no evidence, but format/length instructions matched to setting B (removes the prompt confound)
VERIFY_NO_EVIDENCE_MATCHED = '''Claim:
{claim}

Task:
1. State the label using your own knowledge. Exactly one of: SUPPORT, CONTRADICT, NEI.
   - SUPPORT: the claim is true.
   - CONTRADICT: the claim is false.
   - NEI: you cannot decide whether the claim is true or false.
2. Explain your verdict in 1-3 sentences.

Respond with ONLY a JSON object on one line, no markdown:
{{"label": "<SUPPORT|CONTRADICT|NEI>", "explanation": "<your explanation>"}}'''

write_json(OUT / "prompts.json", {"system": SYSTEM_VERIFY,
                                  "A_no_retrieval": VERIFY_NO_EVIDENCE,
                                  "A2_no_retrieval_matched": VERIFY_NO_EVIDENCE_MATCHED,
                                  "B_with_retrieval": VERIFY_WITH_EVIDENCE})

print(f"Loading verifier LLM: {LLM_MODEL}")
_tok = AutoTokenizer.from_pretrained(LLM_MODEL)
if _tok.pad_token_id is None: _tok.pad_token = _tok.eos_token
_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype="auto", device_map="auto")
_llm.eval()

@torch.no_grad()
def llm_chat(system, user, max_new_tokens=MAX_NEW_TOKENS):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _tok(prompt, return_tensors="pt").to(_llm.device)
    out = _llm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                        pad_token_id=_tok.pad_token_id)
    return _tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

_LABEL_RE = re.compile(r"\b(SUPPORT|CONTRADICT|NEI)\b", re.IGNORECASE)

def parse_response2(text):
    """Returns (label, explanation, parse_mode). parse_mode in {json, regex, FALLBACK_NEI}."""
    c = text.strip().strip("`"); c = re.sub(r"^json", "", c, flags=re.I).strip()
    m = re.search(r"\{.*\}", c, flags=re.DOTALL)
    if m:
        try:
            o = json.loads(m.group(0)); lab = str(o.get("label", "")).upper().strip()
            if lab in LABELS: return lab, str(o.get("explanation", "")).strip(), "json"
        except json.JSONDecodeError: pass
    m = _LABEL_RE.search(c)
    if m: return m.group(1).upper(), c, "regex"
    return "NEI", c, "FALLBACK_NEI"

def extract_summary(text):
    """Pull evidence_summary out of the raw response; '' if absent/unparseable."""
    try:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        return str(json.loads(m.group(0)).get("evidence_summary", "")).strip()
    except Exception:
        return ""

def format_evidence(doc_ids, k):
    blocks, sents = [], []
    for rank, d in enumerate(doc_ids[:k], 1):
        doc = corpus_by_id.get(d)
        if not doc: continue
        sents.extend(doc["sentences"])
        blocks.append(f"[{rank}] {doc['title']}\n{' '.join(doc['sentences'])}")
    return "\n\n".join(blocks), sents

def _record(ex, rdocs, ev_sents, summary, raw):
    lab, expl, mode = parse_response2(raw)
    return {"claim_id": ex["claim_id"], "claim": ex["claim"],
            "gold_label": ex["label"], "gold_doc_ids": ex["gold_doc_ids"],
            "retrieved_doc_ids": rdocs, "evidence_sentences": ev_sents,
            "evidence_summary": summary,
            "pred_label": lab, "explanation": expl, "parse_mode": mode, "raw": raw}

# ----------------------- Setting A: no retrieval (original prompt) -----------------------
pred_noret = []
for ex in tqdm(eval_examples, desc="LLM Setting A (no retrieval)"):
    raw = llm_chat(SYSTEM_VERIFY, VERIFY_NO_EVIDENCE.format(claim=ex["claim"]))
    pred_noret.append(_record(ex, [], [], "", raw))

# ----------------------- Setting A2: no retrieval, prompt matched to B -----------------------
pred_noret_m = []
for ex in tqdm(eval_examples, desc="LLM Setting A2 (no retrieval, matched prompt)"):
    raw = llm_chat(SYSTEM_VERIFY, VERIFY_NO_EVIDENCE_MATCHED.format(claim=ex["claim"]))
    pred_noret_m.append(_record(ex, [], [], "", raw))

# ----------------------- Setting B: with retrieval -----------------------
cache = retrieved_cache[RETRIEVER_FOR_LLM]
pred_ret = []
for ex in tqdm(eval_examples, desc=f"LLM Setting B ({RETRIEVER_FOR_LLM})"):
    rdocs = cache.get(ex["claim_id"], [])
    ev_text, ev_sents = format_evidence(rdocs, RETRIEVAL_K_FOR_LLM)
    raw = llm_chat(SYSTEM_VERIFY, VERIFY_WITH_EVIDENCE.format(
        claim=ex["claim"], evidence=ev_text or "(no evidence retrieved)"))
    pred_ret.append(_record(ex, rdocs[:RETRIEVAL_K_FOR_LLM], ev_sents, extract_summary(raw), raw))

write_jsonl(OUT / "predictions_noretrieval.jsonl", pred_noret)
write_jsonl(OUT / "predictions_noretrieval_matched.jsonl", pred_noret_m)
write_jsonl(OUT / f"predictions_{RETRIEVER_FOR_LLM}.jsonl", pred_ret)

# ----------------------- health checks (report these in the paper) -----------------------
parse_stats = {name: dict(Counter(r["parse_mode"] for r in rows_))
               for name, rows_ in [("A", pred_noret), ("A2", pred_noret_m), ("B", pred_ret)]}
write_json(OUT / "parse_stats.json", parse_stats)
print("parse modes:", parse_stats)          # FALLBACK_NEI silently predicts NEI -> must be ~0

_n_sum = sum(1 for r in pred_ret if r["evidence_summary"])
_n_bad = sum(1 for r in pred_ret if not r["explanation"])
print("generated:", len(pred_noret), "A /", len(pred_noret_m), "A2 /", len(pred_ret), "B")
print(f"evidence_summary populated: {_n_sum}/{len(pred_ret)}  ({100*_n_sum/max(len(pred_ret),1):.0f}%)")
print(f"empty explanations (B): {_n_bad}")
if pred_ret:
    s = pred_ret[0]
    print("\n--- sample Setting B output ---")
    print("claim   :", s["claim"][:100])
    print("label   :", s["pred_label"], "(gold:", s["gold_label"] + ")")
    print("summary :", s["evidence_summary"][:150])
    print("explain :", s["explanation"][:200])

Loading verifier LLM: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM Setting A2 (no retrieval, matched prompt):   0%|          | 0/300 [00:00<?, ?it/s]

LLM Setting B (hybrid):   0%|          | 0/300 [00:00<?, ?it/s]

parse modes: {'A': {'json': 300}, 'A2': {'json': 300}, 'B': {'json': 300}}
generated: 300 A / 300 A2 / 300 B
evidence_summary populated: 300/300  (100%)
empty explanations (B): 0

--- sample Setting B output ---
claim   : 0-dimensional biomaterials show inductive properties.
label   : NEI (gold: NEI)
summary : The provided evidence discusses various biological processes and does not specifically mention 0-dimensional biomaterials or their inductive propertie
explain : The evidence focuses on topics such as bone formation, stem cell differentiation, and neural prosthetics, but does not address 0-dimensional biomaterials or their inductive properties.


## 9. Faithfulness evaluation — the novelty

Prediction accuracy / macro-F1, plus Evidence Coverage, Citation Rate, and
Hallucination Rate per explanation. Each setting is scored against `self` (the
evidence the model saw) and `gold` (common reference for a fair A-vs-B chart).

In [9]:
from sklearn.metrics import accuracy_score, f1_score

if "nanmean" not in globals():            # safety net if cell 6 wasn't updated
    def nanmean(x):
        x = np.asarray(x, float); x = x[~np.isnan(x)]
        return float(x.mean()) if len(x) else float("nan")
nli = None
if USE_NLI:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer as _AT
    print(f"Loading NLI model: {NLI_MODEL}")
    class NLIScorer:
        def __init__(self, name, bs=16):
            self.tok = _AT.from_pretrained(name)
            self.model = AutoModelForSequenceClassification.from_pretrained(name)
            self.model.eval()
            self.dev = "cuda" if torch.cuda.is_available() else "cpu"
            self.model.to(self.dev); self.bs = bs
            id2 = {i: l.lower() for i, l in self.model.config.id2label.items()}
            self.entail = next((i for i, l in id2.items() if "entail" in l), 0)
        @torch.no_grad()
        def probs(self, prem, hyp):
            out = []
            for s in range(0, len(prem), self.bs):
                enc = self.tok(prem[s:s+self.bs], hyp[s:s+self.bs], truncation=True,
                               padding=True, max_length=512, return_tensors="pt").to(self.dev)
                p = torch.softmax(self.model(**enc).logits, -1)[:, self.entail]
                out.extend(p.cpu().tolist())
            return out
        def max_entail(self, hyp, ev):
            if not ev: return 0.0
            prem = list(ev) + [" ".join(ev)]
            return max(self.probs(prem, [hyp]*len(prem)))
    nli = NLIScorer(NLI_MODEL)

def gold_ev_sents(gold_doc_ids):
    s = []
    for d in gold_doc_ids:
        doc = corpus_by_id.get(str(d))
        if doc: s.extend(doc["sentences"])
    return s

def lexical_supported(sent, ev):
    return any(jaccard(sent, e) >= LEXICAL_THRESH for e in ev)

def sentence_scores(sent, ev):
    if not ev: return 0.0, 0.0
    mj = max(jaccard(sent, e) for e in ev)
    me = nli.max_entail(sent, ev) if nli is not None else 0.0
    return mj, me

def sent_table(rows, source, tag):
    recs = []
    for r in tqdm(rows, desc=f"sent-scores[{tag}/{source}]", leave=False):
        ev = r["evidence_sentences"] if source == "self" else gold_ev_sents(r["gold_doc_ids"])
        for j, s in enumerate(split_sentences(r.get("explanation", ""))):
            mj, me = sentence_scores(s, ev)
            recs.append({"claim_id": r["claim_id"], "tag": tag, "source": source,
                         "sent_idx": j, "sentence": s, "max_jac": mj, "max_ent": me})
    write_jsonl(OUT / f"sentscores_{tag}_{source}.jsonl", recs)
    return recs

def per_claim_faith(recs, claim_ids, lex_t=LEXICAL_THRESH, ent_t=FAITH_ENTAIL_THRESH, mode="hybrid"):
    from collections import defaultdict
    by = defaultdict(list)
    for x in recs: by[x["claim_id"]].append(x)
    out = {}
    for cid in claim_ids:
        ss = by.get(cid, [])
        if not ss: out[cid] = (np.nan, np.nan, np.nan); continue      # empty explanation -> NaN, not 0
        lex = [x["max_jac"] >= lex_t for x in ss]
        ent = [x["max_ent"] >= ent_t for x in ss]
        sup = {"hybrid": [a or b for a, b in zip(lex, ent)], "lex": lex, "nli": ent}[mode]
        out[cid] = (np.mean(lex), np.mean(sup), 1 - np.mean(sup))
    return out

def evaluate(rows, source, tag):
    recs = sent_table(rows, source, tag)
    ids = [r["claim_id"] for r in rows]
    pc = per_claim_faith(recs, ids)
    yt = [r["gold_label"] for r in rows]; yp = [r["pred_label"] for r in rows]
    per = [{"claim_id": r["claim_id"], "gold_label": r["gold_label"], "pred_label": r["pred_label"],
            "evidence_coverage": pc[r["claim_id"]][0], "citation_rate": pc[r["claim_id"]][1],
            "hallucination_rate": pc[r["claim_id"]][2]} for r in rows]
    agg = {"n": len(rows), "n_empty_explanations": int(sum(np.isnan(p["citation_rate"]) for p in per)),
           "accuracy": float(accuracy_score(yt, yp)),
           "macro_f1": float(f1_score(yt, yp, labels=LABELS, average="macro", zero_division=0)),
           "evidence_coverage": nanmean([p["evidence_coverage"] for p in per]),
           "citation_rate": nanmean([p["citation_rate"] for p in per]),
           "hallucination_rate": nanmean([p["hallucination_rate"] for p in per])}
    return agg, per

faith = {}
for tag, rows in [("noretrieval", pred_noret), ("noretrieval_matched", pred_noret_m), (RETRIEVER_FOR_LLM, pred_ret)]:
    for source in ["self", "gold"]:
        agg, per = evaluate(rows, source, tag)
        faith[f"{tag}_{source}"] = agg
        write_json(OUT / f"faithfulness_{tag}_{source}.json", agg)
        write_jsonl(OUT / f"faithfulness_{tag}_{source}_perrow.jsonl", per)

import pandas as pd
display(pd.DataFrame(faith).T)

Loading NLI model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (7 outputs left out)

sent-scores[noretrieval_matched/gold]:   0%|          | 0/300 [00:00<?, ?it/s]

sent-scores[hybrid/self]:   0%|          | 0/300 [00:00<?, ?it/s]

sent-scores[hybrid/gold]:   0%|          | 0/300 [00:00<?, ?it/s]

,n,n_empty_explanations,accuracy,macro_f1,evidence_coverage,citation_rate,hallucination_rate
noretrieval_self,300.0,0.0,0.396667,0.386418,0.000000,0.000000,1.000000
noretrieval_gold,300.0,0.0,0.396667,0.386418,0.015000,0.353333,0.646667
noretrieval_matched_self,300.0,0.0,0.410000,0.379607,0.000000,0.000000,1.000000
noretrieval_matched_gold,300.0,0.0,0.410000,0.379607,0.006667,0.413889,0.586111
hybrid_self,300.0,0.0,0.546667,0.535739,0.297778,0.711111,0.288889
hybrid_gold,300.0,0.0,0.546667,0.535739,0.226667,0.541667,0.458333


## 10. Error analysis

Categorizes every claim by retrieval hit/miss × prediction correct/wrong ×
explanation faithful/hallucinated, on the retrieval setting.

In [10]:
faith_self = {r["claim_id"]: r for r in read_jsonl(OUT/f"faithfulness_{RETRIEVER_FOR_LLM}_self_perrow.jsonl")}
faith_gold = {r["claim_id"]: r for r in read_jsonl(OUT/f"faithfulness_{RETRIEVER_FOR_LLM}_gold_perrow.jsonl")}

def hit_k(gold, ret, k): return bool(set(map(str, gold)) & set(map(str, ret[:k])))

def categorize(ref, k, th):
    F = faith_self if ref == "self" else faith_gold
    c = Counter()
    for r in pred_ret:
        h = F[r["claim_id"]]["hallucination_rate"]
        if np.isnan(h): continue
        c[(("HIT" if hit_k(r["gold_doc_ids"], r["retrieved_doc_ids"], k) else "MISS"),
           ("OK" if r["pred_label"] == r["gold_label"] else "WRONG"),
           ("FAITHFUL" if h <= th else "HALLUC"))] += 1
    return c

counts = categorize("self", 5, HAL_TH)
tot = sum(counts.values()); print("claims categorised:", tot, "(should be 300 unless some explanations were empty)")
tbl = pd.DataFrame([{"category": f"retrieval={a} | prediction={b} | explanation={c}",
                     "count": n, "percent": f"{100*n/tot:.1f}%"} for (a, b, c), n in counts.most_common()])
tbl.to_csv(OUT / f"error_analysis_{RETRIEVER_FOR_LLM}.csv", index=False); display(tbl)

sens = []
for ref in ["self", "gold"]:
    for k in [1, 3, 5]:
        for th in [0.3, 0.5, 0.7]:
            c = categorize(ref, k, th); tot = sum(c.values())
            sens.append({"ref": ref, "hit@k": k, "halluc_th": th,
                         "HIT|WRONG|FAITHFUL_%": 100 * c[("HIT", "WRONG", "FAITHFUL")] / tot})
pd.DataFrame(sens).to_csv(OUT / "error_analysis_sensitivity.csv", index=False)

# Export the main error cell for manual coding of failure types
manual = []
for r in pred_ret:
    f = faith_self[r["claim_id"]]
    if (hit_k(r["gold_doc_ids"], r["retrieved_doc_ids"], 5) and r["pred_label"] != r["gold_label"]
            and f["hallucination_rate"] <= HAL_TH):
        manual.append({"claim_id": r["claim_id"], "claim": r["claim"], "gold": r["gold_label"],
                       "pred": r["pred_label"], "gold_rank": next((i+1 for i,d in enumerate(r["retrieved_doc_ids"])
                                                                   if d in set(map(str, r["gold_doc_ids"]))), None),
                       "explanation": r["explanation"],
                       "error_type": "",   # annotate: polarity | NEI_vs_evidence | partial_evidence | label_noise | metric_false_faithful
                       "notes": ""})
pd.DataFrame(manual).to_csv(OUT / "error_cases_to_annotate.csv", index=False)
print(len(manual), "cases to annotate (paper: 82)")

claims categorised: 300 (should be 300 unless some explanations were empty)


,category,count,percent
0,retrieval=HIT | prediction=OK | explanation=FA...,112,37.3%
1,retrieval=HIT | prediction=WRONG | explanation...,83,27.7%
2,retrieval=MISS | prediction=OK | explanation=F...,27,9.0%
3,retrieval=HIT | prediction=WRONG | explanation...,25,8.3%
4,retrieval=MISS | prediction=WRONG | explanatio...,21,7.0%
5,retrieval=HIT | prediction=OK | explanation=HA...,14,4.7%
6,retrieval=MISS | prediction=OK | explanation=H...,11,3.7%
7,retrieval=MISS | prediction=WRONG | explanatio...,7,2.3%


83 cases to annotate (paper: 82)


## 11. Figures (Day 6)

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _box(ax, data, labels):
    try: ax.boxplot(data, tick_labels=labels, showmeans=True)
    except TypeError: ax.boxplot(data, labels=labels, showmeans=True)   # older matplotlib

# panel (a):  _box(ax[0], data, ["HIT", "MISS"])
# panel (b):  _box(ax[1], data, ["HIT", "MISS"])

def _boot_ci_mean(v, n_boot=N_BOOT, seed=SEED):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), (n_boot, len(v)))].mean(axis=1)
    return v.mean(), *np.percentile(means, [2.5, 97.5])

# ---------------- Figure 2: retrieval comparison (95% bootstrap CIs) ----------------
methods = [m for m in ("bm25", "dense", "hybrid") if (OUT / f"retrieval_perclaim_{m}.jsonl").exists()]
rp = {m: list(read_jsonl(OUT / f"retrieval_perclaim_{m}.jsonl")) for m in methods}
series = [f"recall@{k}" for k in TOP_K_VALUES] + ["rr"]
xl = [f"R@{k}" for k in TOP_K_VALUES] + ["MRR"]
x = np.arange(len(series)); w = 0.8 / max(len(methods), 1)
fig, ax = plt.subplots(figsize=(7, 4.2))
for i, m in enumerate(methods):
    st = np.array([_boot_ci_mean([r[s] for r in rp[m]]) for s in series])
    ax.bar(x + i * w, st[:, 0], w, label=m.upper(),
           yerr=[st[:, 0] - st[:, 1], st[:, 2] - st[:, 0]], capsize=2, error_kw={"lw": 0.8})
ax.set_xticks(x + w * (len(methods) - 1) / 2); ax.set_xticklabels(xl)
ax.set_ylabel("Score"); ax.set_ylim(0, 1); ax.set_title("Retrieval quality on SciFact (95% CI)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for ext in ("png", "pdf"): fig.savefig(OUT / f"figure2_retrieval_comparison.{ext}", bbox_inches="tight", dpi=200)
plt.show()

# ---------------- Figure 3: faithfulness vs gold (95% CIs, incl. prompt-matched baseline) ----------------
cands = [("noretrieval", "No retrieval"),
         ("noretrieval_matched", "No retrieval (matched prompt)"),
         (RETRIEVER_FOR_LLM, "With retrieval")]
cands = [(t, n_) for t, n_ in cands if (OUT / f"faithfulness_{t}_gold_perrow.jsonl").exists()]
mets   = ["evidence_coverage", "citation_rate", "hallucination_rate"]
labels = ["Evidence\nCoverage", "Citation\nRate", "Hallucination\nRate"]
x = np.arange(len(mets)); w = 0.8 / len(cands)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, (t, n_) in enumerate(cands):
    per = list(read_jsonl(OUT / f"faithfulness_{t}_gold_perrow.jsonl"))
    st = np.array([_boot_ci_mean([r[m] for r in per]) for m in mets])
    ax.bar(x + i * w - 0.4 + w / 2, st[:, 0], w, label=n_,
           yerr=[st[:, 0] - st[:, 1], st[:, 2] - st[:, 0]], capsize=2, error_kw={"lw": 0.8})
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.set_title("Explanation faithfulness (reference: gold evidence, 95% CI)")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for ext in ("png", "pdf"): fig.savefig(OUT / f"figure3_faithfulness_comparison.{ext}", bbox_inches="tight", dpi=200)
plt.show()

## 12. Summary of outputs

In [12]:
print("Files written to", OUT, ":\n")
for p in sorted(OUT.glob("*")):
    print("  ", p.name)

print("\n--- Run configuration (config.json) ---")
cfg = json.load(open(OUT / "config.json"))
for k_ in ("LLM_MODEL", "DENSE_ENCODER", "NLI_MODEL", "SEED", "MAX_NEW_TOKENS", "DECODING", "N_BOOT"):
    print(f"  {k_}: {cfg.get(k_)}")
print("  versions:", cfg.get("versions"))

print("\n--- Generation health ---")
print("  parse modes (FALLBACK_NEI must be ~0):", json.load(open(OUT / "parse_stats.json")))
for k_, v in faith.items():
    if k_.endswith("_gold"):
        print(f"  empty explanations [{k_}]: {v['n_empty_explanations']}")

print("\n--- Headline numbers ---")
print("  Retrieval MRR :", {m: round(retrieval_metrics[m]['mrr'], 3) for m in retrieval_metrics})
print("  Retrieval R@5 :", {m: round(retrieval_metrics[m]['recall@5'], 3) for m in retrieval_metrics})
a  = faith["noretrieval_gold"]; b = faith[f"{RETRIEVER_FOR_LLM}_gold"]
print(f"  Accuracy      : no-ret = {a['accuracy']:.3f} | ret = {b['accuracy']:.3f}")
print(f"  Citation (gold)      : no-ret = {a['citation_rate']:.3f} | ret = {b['citation_rate']:.3f}")
print(f"  Hallucination (gold) : no-ret = {a['hallucination_rate']:.3f} | ret = {b['hallucination_rate']:.3f}")
if "noretrieval_matched_gold" in faith:
    m_ = faith["noretrieval_matched_gold"]
    print(f"  Matched-prompt baseline (A2): acc = {m_['accuracy']:.3f}, cit = {m_['citation_rate']:.3f}, "
          f"hall = {m_['hallucination_rate']:.3f}")

# Sanity check vs the submitted paper (Table II), only meaningful for the original verifier
if LLM_TAG == "Qwen2.5-3B-Instruct":
    paper = {"accuracy": (0.397, 0.550), "citation_rate": (0.353, 0.543), "hallucination_rate": (0.647, 0.457)}
    for k_, (pa, pb) in paper.items():
        ok = abs(a[k_] - pa) < 0.01 and abs(b[k_] - pb) < 0.01
        print(f"  [{'OK' if ok else 'DIFFERS'}] {k_}: paper {pa}/{pb} vs now {a[k_]:.3f}/{b[k_]:.3f}")
    print("  If something DIFFERS, check empty explanations first (they are NaN now, previously counted as 0 in per-row files).")

Files written to /kaggle/working/Qwen2.5-3B-Instruct :

   config.json
   corpus.jsonl
   error_analysis_hybrid.csv
   error_analysis_sensitivity.csv
   error_cases_to_annotate.csv
   eval_examples.jsonl
   faithfulness_hybrid_gold.json
   faithfulness_hybrid_gold_perrow.jsonl
   faithfulness_hybrid_self.json
   faithfulness_hybrid_self_perrow.jsonl
   faithfulness_noretrieval_gold.json
   faithfulness_noretrieval_gold_perrow.jsonl
   faithfulness_noretrieval_matched_gold.json
   faithfulness_noretrieval_matched_gold_perrow.jsonl
   faithfulness_noretrieval_matched_self.json
   faithfulness_noretrieval_matched_self_perrow.jsonl
   faithfulness_noretrieval_self.json
   faithfulness_noretrieval_self_perrow.jsonl
   figure2_retrieval_comparison.pdf
   figure2_retrieval_comparison.png
   figure3_faithfulness_comparison.pdf
   figure3_faithfulness_comparison.png
   parse_stats.json
   predictions_hybrid.jsonl
   predictions_noretrieval.jsonl
   predictions_noretrieval_matched.jsonl
   promp

In [13]:
# import glob, json
# from pathlib import Path

# def _find(name):
#     hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)) \
#         or sorted(glob.glob(f"./**/{name}", recursive=True))
#     return hits[0] if hits else None

# def read_jsonl(p):
#     with open(p, encoding="utf-8") as fh:
#         for line in fh:
#             line = line.strip()
#             if line: yield json.loads(line)

# CORPUS = _find("corpus.jsonl")
# DEV    = _find("claims_dev.jsonl")
# TRAIN  = _find("claims_train.jsonl")
# print("corpus:", CORPUS, "\ndev:", DEV, "\ntrain:", TRAIN, "\n")

# # ---- one raw CORPUS document ----
# doc = next(read_jsonl(CORPUS))
# print("="*70, "\nCORPUS DOCUMENT (one scientific abstract)\n", "="*70)
# print(json.dumps(doc, indent=2)[:900], "...\n")
# print("Fields:")
# print("  doc_id     -> unique abstract id (claims cite these)")
# print("  title      -> paper title")
# print("  abstract   -> LIST of sentences (already sentence-split); evidence")
# print("                points at these by index")
# print("  structured -> whether the abstract had section headers\n")

# # ---- a few raw CLAIMS, one of each label ----
# print("="*70, "\nCLAIMS (what the model must verify)\n", "="*70)
# seen = set()
# for c in read_jsonl(DEV):
#     ev = c.get("evidence", {}) or {}
#     labels = [it["label"] for v in ev.values() for it in v]
#     lab = "SUPPORT" if "SUPPORT" in labels else \
#           "CONTRADICT" if "CONTRADICT" in labels else "NEI"
#     if lab in seen: continue
#     seen.add(lab)
#     print(f"\n--- example with derived label = {lab} ---")
#     print(json.dumps(c, indent=2))
#     if len(seen) == 3: break

# print("""
# Field guide for a claim:
#   id            -> claim id
#   claim         -> the sentence to be checked
#   cited_doc_ids -> abstract(s) a human consulted  == retrieval GOLD target
#   evidence      -> {doc_id: [{"sentences":[idx...], "label": SUPPORT|CONTRADICT}]}
#                    - which sentences justify the verdict, and the direction
#                    - EMPTY {}  ==  Not Enough Info (NEI)
# Label derivation used in the pipeline:
#   any SUPPORT -> SUPPORT ; else any CONTRADICT -> CONTRADICT ; else NEI
# """)

In [14]:
# =====================================================================
#  Correlation analysis
#  Does per-claim retrieval quality predict per-claim explanation faithfulness?
#    Q1: higher Recall@5      -> lower hallucination?
#    Q2: higher MRR (rank)    -> higher citation rate?
#  Computed against BOTH references:
#    gold  = faithfulness vs the human-annotated evidence  (the real test)
#    self  = faithfulness vs whatever was retrieved        (control/contrast)
#  The self-vs-gold contrast is itself a result: retrieval quality should
#  predict GOLD faithfulness strongly but SELF faithfulness weakly, because a
#  retrieval MISS yields an explanation faithfully grounded in the WRONG doc.
# =====================================================================
import json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr, mannwhitneyu

# --- run standalone or inside the notebook: recover OUT / read_jsonl ---
try: OUT
except NameError:
    OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
try: read_jsonl
except NameError:
    def read_jsonl(p):
        with open(p, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line: yield json.loads(line)

METHOD = globals().get("RETRIEVER_FOR_LLM", "hybrid")
K      = globals().get("RETRIEVAL_K_FOR_LLM", 5)
def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"

# ---------- 1. per-claim retrieval quality (from predictions file) ----------
preds = {r["claim_id"]: r for r in read_jsonl(OUT / f"predictions_{METHOD}.jsonl")}

def recall_at_k(ret, gold, k):
    return len(set(ret[:k]) & set(gold)) / len(gold) if gold else np.nan
def reciprocal_rank(ret, gold):
    g = set(gold)
    for i, d in enumerate(ret, 1):
        if d in g: return 1.0 / i
    return 0.0

rows = []
for cid, r in preds.items():
    gold = [str(x) for x in r["gold_doc_ids"]]
    ret  = [str(x) for x in r["retrieved_doc_ids"]]
    if not gold:            # retrieval metric undefined when nothing is cited
        continue
    rows.append({"claim_id": cid, "gold_label": r["gold_label"],
                 "recall_at_5": recall_at_k(ret, gold, K),
                 "reciprocal_rank": reciprocal_rank(ret, gold),
                 "retrieval_hit": int(any(d in set(gold) for d in ret))})
retr_df = pd.DataFrame(rows).set_index("claim_id")

# ---------- 2. per-claim faithfulness (self + gold references) ----------
def load_faith(ref):
    recs = list(read_jsonl(OUT / f"faithfulness_{METHOD}_{ref}_perrow.jsonl"))
    return pd.DataFrame([{"claim_id": v["claim_id"],
                          f"citation_{ref}": v["citation_rate"],
                          f"halluc_{ref}":  v["hallucination_rate"]}
                         for v in recs]).set_index("claim_id")

df = retr_df.join(load_faith("self")).join(load_faith("gold"))
df = df.dropna(subset=["citation_gold", "halluc_gold"])
print(f"Claims in correlation analysis: {len(df)} "
      f"(retrieval hits: {int(df.retrieval_hit.sum())}, "
      f"misses: {int((1-df.retrieval_hit).sum())})\n")

# ---------- 3. correlation table ----------
def corr(x, y, xn, yn):
    m = x.notna() & y.notna(); xs, ys = x[m], y[m]
    sr, sp = spearmanr(xs, ys); pr, pp = pearsonr(xs, ys)
    return {"retrieval_metric": xn, "faithfulness_metric": yn,
            "spearman_r": sr, "spearman_p": sp, "sig": stars(sp),
            "pearson_r": pr, "pearson_p": pp, "n": int(m.sum())}

pairs = []
for rq, rqn in [("recall_at_5", "Recall@5"),
                ("reciprocal_rank", "Reciprocal Rank")]:
    for fm, fmn in [("halluc_gold",   "Hallucination (vs gold)"),
                    ("citation_gold", "Citation rate (vs gold)"),
                    ("halluc_self",   "Hallucination (vs retrieved)"),
                    ("citation_self", "Citation rate (vs retrieved)")]:
        pairs.append(corr(df[rq], df[fm], rqn, fmn))
corr_df = pd.DataFrame(pairs)
corr_df.to_csv(OUT / "correlation_analysis.csv", index=False)
print("=== Correlations (Spearman is primary; retrieval vars are discrete) ===")
# after corr_df is built:
corr_df["spearman_p_holm"] = holm(corr_df["spearman_p"].values)   # holm() defined in the new cell below; paste it above this cell

hit_df = df[df.retrieval_hit == 1]
within = pd.DataFrame([corr(hit_df["reciprocal_rank"], hit_df[c], "RR (HIT only)", c)
                       for c in ["halluc_gold", "citation_gold"]])
within.to_csv(OUT / "correlation_within_hits.csv", index=False)
print("Does rank still matter once the gold doc is in the top-5?"); print(within.round(3))
try: display(corr_df.round(3))
except NameError: print(corr_df.round(3).to_string(index=False))

# ---------- 4. HIT vs MISS group comparison (cleaner for a ~binary cause) ----------
def group_test(col):
    hit  = df.loc[df.retrieval_hit == 1, col].dropna()
    miss = df.loc[df.retrieval_hit == 0, col].dropna()
    if len(hit) == 0 or len(miss) == 0: return None
    u, p = mannwhitneyu(hit, miss, alternative="two-sided")
    rank_biserial = 1 - 2 * u / (len(hit) * len(miss))   # effect size
    return {"metric": col, "hit_mean": hit.mean(), "miss_mean": miss.mean(),
            "delta": hit.mean() - miss.mean(), "n_hit": len(hit),
            "n_miss": len(miss), "mannwhitney_p": p,
            "rank_biserial": rank_biserial, "sig": stars(p)}

grp = pd.DataFrame([g for g in (group_test(c) for c in
      ["halluc_gold", "citation_gold", "halluc_self", "citation_self"]) if g])
print("\n=== Retrieval HIT vs MISS (Mann-Whitney U) ===")
try: display(grp.round(3))
except NameError: print(grp.round(3).to_string(index=False))

# ---------- 5. visualization ----------
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) hallucination vs gold, by hit/miss
data = [df.loc[df.retrieval_hit == 1, "halluc_gold"].dropna(),
        df.loc[df.retrieval_hit == 0, "halluc_gold"].dropna()]
ax[0].boxplot(data, labels=["HIT", "MISS"], showmeans=True)
ax[0].set_title("Hallucination vs GOLD\n(retrieval hit vs miss)")
ax[0].set_ylabel("hallucination rate"); ax[0].grid(axis="y", alpha=0.3)

# (b) hallucination vs self, by hit/miss  -> the contrast
data = [df.loc[df.retrieval_hit == 1, "halluc_self"].dropna(),
        df.loc[df.retrieval_hit == 0, "halluc_self"].dropna()]
ax[1].boxplot(data, labels=["HIT", "MISS"], showmeans=True)
ax[1].set_title("Hallucination vs RETRIEVED\n(should differ far less)")
ax[1].set_ylabel("hallucination rate"); ax[1].grid(axis="y", alpha=0.3)

# (c) mean gold hallucination by reciprocal-rank bucket (the graded view)
by_rr = df.groupby("reciprocal_rank")["halluc_gold"].agg(["mean", "count"])
ax[2].bar(np.arange(len(by_rr)), by_rr["mean"])
ax[2].set_xticks(np.arange(len(by_rr)))
ax[2].set_xticklabels([f"{r:.2f}\n(n={int(c)})"
                       for r, c in zip(by_rr.index, by_rr["count"])], fontsize=8)
ax[2].set_title("Hallucination vs GOLD\nby reciprocal rank of first gold doc")
ax[2].set_xlabel("reciprocal rank (0 = missed)"); ax[2].set_ylabel("mean hallucination")
ax[2].grid(axis="y", alpha=0.3)
plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(OUT / f"figure4_correlation.{ext}", bbox_inches="tight", dpi=200)
plt.show()

# ---------- 6. auto-written interpretation ----------
def get(rqn, fmn, col="spearman_r"):
    row = corr_df[(corr_df.retrieval_metric == rqn) &
                  (corr_df.faithfulness_metric == fmn)].iloc[0]
    return row[col], row["spearman_p"]

r1, p1 = get("Recall@5", "Hallucination (vs gold)")
r2, p2 = get("Reciprocal Rank", "Citation rate (vs gold)")
r1s, _ = get("Recall@5", "Hallucination (vs retrieved)")

def word(r): return "negatively" if r < 0 else "positively"
print("\n" + "=" * 68)
print("INTERPRETATION")
print("=" * 68)
print(f"Q1  Recall@5 vs hallucination(gold): rho={r1:+.3f} ({stars(p1)}). "
      f"Higher recall is {word(r1)} associated with hallucination"
      f"{' -> retrieval quality predicts more faithful explanations.' if r1 < 0 else '.'}")
print(f"Q2  Reciprocal rank vs citation(gold): rho={r2:+.3f} ({stars(p2)}). "
      f"Ranking the right doc higher is {word(r2)} associated with citation rate"
      f"{' -> better ranking predicts better-grounded explanations.' if r2 > 0 else '.'}")
print(f"\nContrast: Recall@5 vs hallucination is rho={r1:+.3f} against GOLD "
      f"but rho={r1s:+.3f} against RETRIEVED evidence.")
print("If the GOLD effect is much stronger, that is the key story: retrieval")
print("misses do not make explanations look unfaithful to the model's OWN")
print("evidence -- the model stays faithful to the WRONG document -- so the")
print("damage only shows up against gold. Retrieval quality and explanation")
print("faithfulness are causally linked, not just co-reported metrics.")

Claims in correlation analysis: 300 (retrieval hits: 234, misses: 66)

=== Correlations (Spearman is primary; retrieval vars are discrete) ===
Does rank still matter once the gold doc is in the top-5?
  retrieval_metric faithfulness_metric  spearman_r  spearman_p  sig  \
0    RR (HIT only)         halluc_gold       -0.24         0.0  ***   
1    RR (HIT only)       citation_gold        0.24         0.0  ***   

   pearson_r  pearson_p    n  
0     -0.243        0.0  234  
1      0.243        0.0  234  


,retrieval_metric,faithfulness_metric,spearman_r,spearman_p,sig,pearson_r,pearson_p,n,spearman_p_holm
0,Recall@5,Hallucination (vs gold),-0.334,0.000,***,-0.357,0.000,300,0.000
1,Recall@5,Citation rate (vs gold),0.334,0.000,***,0.357,0.000,300,0.000
2,Recall@5,Hallucination (vs retrieved),-0.158,0.006,**,-0.161,0.005,300,0.018
3,Recall@5,Citation rate (vs retrieved),0.158,0.006,**,0.161,0.005,300,0.018
4,Reciprocal Rank,Hallucination (vs gold),-0.425,0.000,***,-0.428,0.000,300,0.000
5,Reciprocal Rank,Citation rate (vs gold),0.425,0.000,***,0.428,0.000,300,0.000
6,Reciprocal Rank,Hallucination (vs retrieved),-0.164,0.004,**,-0.161,0.005,300,0.018
7,Reciprocal Rank,Citation rate (vs retrieved),0.164,0.004,**,0.161,0.005,300,0.018



=== Retrieval HIT vs MISS (Mann-Whitney U) ===


,metric,hit_mean,miss_mean,delta,n_hit,n_miss,mannwhitney_p,rank_biserial,sig
0,halluc_gold,0.371,0.768,-0.397,234,66,0.000,0.489,***
1,citation_gold,0.629,0.232,0.397,234,66,0.000,-0.489,***
2,halluc_self,0.252,0.419,-0.167,234,66,0.001,0.229,**
3,citation_self,0.748,0.581,0.167,234,66,0.001,-0.229,**


/tmp/ipykernel_22/4275412446.py:126: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax[0].boxplot(data, labels=["HIT", "MISS"], showmeans=True)
/tmp/ipykernel_22/4275412446.py:133: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax[1].boxplot(data, labels=["HIT", "MISS"], showmeans=True)



INTERPRETATION
Q1  Recall@5 vs hallucination(gold): rho=-0.334 (***). Higher recall is negatively associated with hallucination -> retrieval quality predicts more faithful explanations.
Q2  Reciprocal rank vs citation(gold): rho=+0.425 (***). Ranking the right doc higher is positively associated with citation rate -> better ranking predicts better-grounded explanations.

Contrast: Recall@5 vs hallucination is rho=-0.334 against GOLD but rho=-0.158 against RETRIEVED evidence.
If the GOLD effect is much stronger, that is the key story: retrieval
misses do not make explanations look unfaithful to the model's OWN
evidence -- the model stays faithful to the WRONG document -- so the
damage only shows up against gold. Retrieval quality and explanation
faithfulness are causally linked, not just co-reported metrics.


In [15]:
# =====================================================================
#  Statistical validation + per-class analysis
#    (a) Bootstrap 95% CIs for headline metrics (A, A2, B)
#    (b) Paired tests B vs A and B vs A2 (McNemar, paired bootstrap, Wilcoxon) + Holm
#    (c) Per-class P/R/F1 + confusion matrices
#    (d) Paired retrieval-method tests from saved per-claim files + Holm
# =====================================================================
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, binomtest
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

assert "OUT" in globals(), "Run cell 4 first (OUT is per-verifier)."
if "read_jsonl" not in globals():
    def read_jsonl(p):
        with open(p, encoding="utf-8") as fh:
            for line in fh:
                if line.strip(): yield json.loads(line)

METHOD = globals().get("RETRIEVER_FOR_LLM", "hybrid")
LABELS = globals().get("LABELS", ["SUPPORT", "CONTRADICT", "NEI"])
K      = globals().get("RETRIEVAL_K_FOR_LLM", 5)
N_BOOT = globals().get("N_BOOT", 5000)
RNG    = np.random.default_rng(globals().get("SEED", 42))
if "holm" not in globals():
    def holm(p):
        p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
        for rank, i in enumerate(o):
            run = max(run, (m - rank) * p[i]); adj[i] = min(run, 1.0)
        return adj

def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
def fmt(m, lo, hi): return f"{m:.3f} [{lo:.3f}, {hi:.3f}]"
def show(d):
    try: display(d)
    except NameError: print(d.to_string(index=False))

print(f"Reading results from: {OUT}\n")

# ---------------- 0. load settings ----------------
SETTINGS = {"A": "noretrieval", "A2": "noretrieval_matched", "B": METHOD}
NAMES    = {"A": "No retrieval", "A2": "No retrieval (matched prompt)", "B": f"Retrieval ({METHOD})"}

def _load(path, key="claim_id"):
    return {r[key]: r for r in read_jsonl(path)} if path.exists() else None

P = {k: _load(OUT / f"predictions_{t}.jsonl") for k, t in SETTINGS.items()}
F = {k: _load(OUT / f"faithfulness_{t}_gold_perrow.jsonl") for k, t in SETTINGS.items()}
avail = [k for k in SETTINGS if P[k] is not None and F[k] is not None]
assert "A" in avail and "B" in avail, f"missing outputs; found settings: {avail}"
print("settings available:", [NAMES[k] for k in avail])

lab2i = {l: i for i, l in enumerate(LABELS)}
ids = sorted(set.intersection(*[set(P[k]) & set(F[k]) for k in avail]))
n = len(ids); print(f"Paired claims available: {n}\n")
yt = np.array([lab2i[P["B"][i]["gold_label"]] for i in ids])
YP = {k: np.array([lab2i[P[k][i]["pred_label"]] for i in ids]) for k in avail}

# ---------------- 1. bootstrap machinery ----------------
def macro_f1_idx(t, p, L=len(LABELS)):
    cm = np.bincount(t * L + p, minlength=L * L).reshape(L, L)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    return float(np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0).mean())

def boot_idx(m): return RNG.integers(0, m, size=(N_BOOT, m))

def ci_mean(v):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    d = v[boot_idx(len(v))].mean(axis=1)
    return (v.mean(), *np.percentile(d, [2.5, 97.5]))

def ci_label(t, p, metric):
    idx = boot_idx(len(t))
    if metric == "acc":
        return ((t == p).mean(), *np.percentile((t[idx] == p[idx]).mean(axis=1), [2.5, 97.5]))
    d = np.array([macro_f1_idx(t[i], p[i]) for i in idx])
    return (macro_f1_idx(t, p), *np.percentile(d, [2.5, 97.5]))

def paired_boot_delta(a, b):
    """Paired bootstrap on mean(b)-mean(a); two-sided p from resamples crossing 0 (floored at 1/N_BOOT)."""
    d = np.asarray(b, float) - np.asarray(a, float)
    dist = d[boot_idx(len(d))].mean(axis=1)
    lo, hi = np.percentile(dist, [2.5, 97.5])
    p = 2 * min((dist <= 0).mean(), (dist >= 0).mean())
    return d.mean(), lo, hi, min(max(p, 1 / N_BOOT), 1.0)

# ---------------- 2. headline metrics ----------------
rows = []
for k in avail:
    g = lambda key: [F[k][i][key] for i in ids]
    rows.append({"setting": NAMES[k],
                 "accuracy": fmt(*ci_label(yt, YP[k], "acc")),
                 "macro_F1": fmt(*ci_label(yt, YP[k], "f1")),
                 "evidence_coverage": fmt(*ci_mean(g("evidence_coverage"))),
                 "citation (gold)": fmt(*ci_mean(g("citation_rate"))),
                 "hallucination (gold)": fmt(*ci_mean(g("hallucination_rate"))),
                 "n_empty_expl": int(np.isnan(np.array(g("citation_rate"), float)).sum())})
ci_df = pd.DataFrame(rows); ci_df.to_csv(OUT / "bootstrap_cis.csv", index=False)
print("=" * 78); print(f"HEADLINE METRICS - point [95% bootstrap CI], n={n}, N_BOOT={N_BOOT}"); print("=" * 78)
show(ci_df)

# ---------------- 3. paired tests: B vs A, B vs A2 ----------------
def compare(kA, kB="B"):
    t, ya, yb = yt, YP[kA], YP[kB]
    ca, cb = (ya == t).astype(int), (yb == t).astype(int)
    b01 = int(((ca == 0) & (cb == 1)).sum()); b10 = int(((ca == 1) & (cb == 0)).sum())
    mc_p = binomtest(b01, b01 + b10, 0.5).pvalue if (b01 + b10) > 0 else 1.0
    d, lo, hi, _ = paired_boot_delta(ca, cb)
    out = [{"metric": "accuracy", "delta": f"{d:+.3f}", "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "test": f"McNemar exact (b01={b01}, b10={b10})", "n": len(t), "p": mc_p}]
    idx = boot_idx(len(t))
    fd = np.array([macro_f1_idx(t[i], yb[i]) - macro_f1_idx(t[i], ya[i]) for i in idx])
    pt = macro_f1_idx(t, yb) - macro_f1_idx(t, ya)
    pf = min(max(2 * min((fd <= 0).mean(), (fd >= 0).mean()), 1 / N_BOOT), 1.0)
    out.append({"metric": "macro_F1", "delta": f"{pt:+.3f}",
                "95% CI": f"[{np.percentile(fd, 2.5):+.3f}, {np.percentile(fd, 97.5):+.3f}]",
                "test": "paired bootstrap", "n": len(t), "p": pf})
    for name, key in [("citation (gold)", "citation_rate"), ("hallucination (gold)", "hallucination_rate")]:
        va = np.array([F[kA][i][key] for i in ids], float); vb = np.array([F[kB][i][key] for i in ids], float)
        m = ~(np.isnan(va) | np.isnan(vb)); va, vb = va[m], vb[m]          # drop empty explanations
        d, lo, hi, _ = paired_boot_delta(va, vb)
        try: w = wilcoxon(vb, va, zero_method="zsplit").pvalue
        except ValueError: w = 1.0
        out.append({"metric": name, "delta": f"{d:+.3f}", "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                    "test": "Wilcoxon signed-rank", "n": int(m.sum()), "p": w})
    df_ = pd.DataFrame(out)
    df_["p_holm"] = holm(df_["p"].values); df_["sig_holm"] = df_["p_holm"].map(stars)
    df_.insert(0, "comparison", f"{NAMES['B']}  vs  {NAMES[kA]}")
    return df_, (b01, b10, mc_p)

sig_parts, mc_info = [], {}
for kA in [k for k in avail if k != "B"]:
    d_, mc_info[kA] = compare(kA); sig_parts.append(d_)
sig_df = pd.concat(sig_parts, ignore_index=True); sig_df.to_csv(OUT / "paired_significance.csv", index=False)
print("\n" + "=" * 78); print("PAIRED TESTS (Holm-adjusted within each comparison)"); print("=" * 78)
show(sig_df)

# ---------------- 4. per-class + confusion ----------------
print("\n" + "=" * 78); print("PER-CLASS BREAKDOWN"); print("=" * 78)
pc = []
for k in avail:
    p_, r_, f_, s_ = precision_recall_fscore_support(yt, YP[k], labels=range(len(LABELS)), zero_division=0)
    for li, lab in enumerate(LABELS):
        pc.append({"setting": NAMES[k], "class": lab, "support": int(s_[li]),
                   "precision": p_[li], "recall": r_[li], "f1": f_[li]})
percls_df = pd.DataFrame(pc); percls_df.to_csv(OUT / "per_class_metrics.csv", index=False); show(percls_df.round(3))
piv = percls_df.pivot(index="class", columns="setting", values="f1")
piv["delta_f1 (B - A)"] = piv[NAMES["B"]] - piv[NAMES["A"]]
print("\nPer-class F1 change:"); show(piv.round(3).reset_index())

for k in ["A", "B"]:      # where do gold SUPPORT / CONTRADICT claims go?  (CONTRADICT recall drops, SUPPORT->CONTRADICT leak)
    cm = confusion_matrix(yt, YP[k], labels=range(len(LABELS)))
    print(f"\nRow-normalised confusion (gold rows -> predicted cols): {NAMES[k]}")
    print(pd.DataFrame(cm / cm.sum(1, keepdims=True), index=LABELS, columns=LABELS).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, k in zip(axes, ["A", "B"]):
    cm = confusion_matrix(yt, YP[k], labels=range(len(LABELS))); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=30, ha="right")
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold")
    ax.set_title(f"{NAMES[k]}\nacc={(yt == YP[k]).mean():.3f}")
    for r_ in range(cm.shape[0]):
        for c_ in range(cm.shape[1]):
            ax.text(c_, r_, cm[r_, c_], ha="center", va="center", color="white" if cm[r_, c_] > cm.max() / 2 else "black")
plt.tight_layout()
for ext in ("png", "pdf"): fig.savefig(OUT / f"figure5_confusion_matrices.{ext}", bbox_inches="tight", dpi=200)
plt.show()

# ---------------- 5. retrieval methods (from saved per-claim files) ----------------
print("\n" + "=" * 78); print("RETRIEVAL METHOD COMPARISON (paired; Holm over the 6-test family)"); print("=" * 78)
rp = {m: _load(OUT / f"retrieval_perclaim_{m}.jsonl") for m in ("bm25", "dense", "hybrid")}
rp = {m: v for m, v in rp.items() if v is not None}
if len(rp) < 2:
    print("retrieval_perclaim_*.jsonl not found -> re-run the retrieval cell (cell 14).")
else:
    cids = sorted(set.intersection(*[set(v) for v in rp.values()]))
    score_rows, delta_rows = [], []
    for key, mname in [(f"recall@{K}", f"Recall@{K}"), ("rr", "MRR")]:
        vals = {m: np.array([rp[m][c][key] for c in cids], float) for m in rp}
        for m in rp:
            score_rows.append({"metric": mname, "method": m, "score": fmt(*ci_mean(vals[m]))})
        for x, y in [("dense", "bm25"), ("hybrid", "bm25"), ("hybrid", "dense")]:
            if x in vals and y in vals:
                d, lo, hi, p = paired_boot_delta(vals[y], vals[x])
                delta_rows.append({"metric": mname, "comparison": f"{x} - {y}", "delta": f"{d:+.3f}",
                                   "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "p": p})
    dd = pd.DataFrame(delta_rows); dd["p_holm"] = holm(dd["p"].values); dd["sig_holm"] = dd["p_holm"].map(stars)
    sc = pd.DataFrame(score_rows)
    sc.to_csv(OUT / "retrieval_scores_ci.csv", index=False); dd.to_csv(OUT / "retrieval_significance.csv", index=False)
    show(sc); show(dd)

# ---------------- 6. auto summary ----------------
print("\n" + "=" * 78); print("INTERPRETATION"); print("=" * 78)
b01, b10, mc_p = mc_info["A"]
ha = np.array([F["A"][i]["hallucination_rate"] for i in ids], float)
hb = np.array([F["B"][i]["hallucination_rate"] for i in ids], float)
m_ = ~(np.isnan(ha) | np.isnan(hb)); hd, hlo, hhi, _ = paired_boot_delta(ha[m_], hb[m_])
print(f"Accuracy      {(yt == YP['A']).mean():.3f} -> {(yt == YP['B']).mean():.3f}  "
      f"(McNemar p={mc_p:.2g}); {b01} claims fixed by retrieval, {b10} broken.")
print(f"Hallucination {np.nanmean(ha):.3f} -> {np.nanmean(hb):.3f}  (delta {hd:+.3f}, 95% CI [{hlo:+.3f}, {hhi:+.3f}])")
best = piv["delta_f1 (B - A)"].idxmax()
print(f"Largest per-class F1 gain from retrieval: {best} ({piv.loc[best, 'delta_f1 (B - A)']:+.3f}).")
if "A2" in avail:
    r_ = sig_df[sig_df.comparison.str.contains("matched")]
    print("\nPrompt-matched baseline (A2) - does the effect survive?"); show(r_[["metric", "delta", "p_holm", "sig_holm"]])

Reading results from: /kaggle/working/Qwen2.5-3B-Instruct

settings available: ['No retrieval', 'No retrieval (matched prompt)', 'Retrieval (hybrid)']
Paired claims available: 300

HEADLINE METRICS - point [95% bootstrap CI], n=300, N_BOOT=5000


,setting,accuracy,macro_F1,evidence_coverage,citation (gold),hallucination (gold),n_empty_expl
0,No retrieval,"0.397 [0.343, 0.453]","0.386 [0.331, 0.440]","0.015 [0.003, 0.028]","0.353 [0.303, 0.403]","0.647 [0.597, 0.697]",0
1,No retrieval (matched prompt),"0.410 [0.357, 0.467]","0.380 [0.324, 0.433]","0.007 [0.000, 0.017]","0.414 [0.359, 0.468]","0.586 [0.531, 0.641]",0
2,Retrieval (hybrid),"0.547 [0.487, 0.603]","0.536 [0.478, 0.590]","0.227 [0.182, 0.272]","0.542 [0.492, 0.592]","0.458 [0.408, 0.510]",0



PAIRED TESTS (Holm-adjusted within each comparison)


,comparison,metric,delta,95% CI,test,n,p,p_holm,sig_holm
0,Retrieval (hybrid) vs No retrieval,accuracy,+0.150,"[+0.083, +0.217]","McNemar exact (b01=80, b10=35)",300,3.284442e-05,6.568884e-05,***
1,Retrieval (hybrid) vs No retrieval,macro_F1,+0.149,"[+0.081, +0.215]",paired bootstrap,300,2.000000e-04,2.000000e-04,***
2,Retrieval (hybrid) vs No retrieval,citation (gold),+0.188,"[+0.126, +0.251]",Wilcoxon signed-rank,300,1.317733e-07,5.270931e-07,***
3,Retrieval (hybrid) vs No retrieval,hallucination (gold),-0.188,"[-0.251, -0.126]",Wilcoxon signed-rank,300,1.317733e-07,5.270931e-07,***
4,Retrieval (hybrid) vs No retrieval (matched ...,accuracy,+0.137,"[+0.070, +0.203]","McNemar exact (b01=76, b10=35)",300,1.246562e-04,4.986250e-04,***
5,Retrieval (hybrid) vs No retrieval (matched ...,macro_F1,+0.156,"[+0.090, +0.223]",paired bootstrap,300,2.000000e-04,6.000000e-04,***
6,Retrieval (hybrid) vs No retrieval (matched ...,citation (gold),+0.128,"[+0.063, +0.192]",Wilcoxon signed-rank,300,4.190608e-04,8.381216e-04,***
7,Retrieval (hybrid) vs No retrieval (matched ...,hallucination (gold),-0.128,"[-0.191, -0.066]",Wilcoxon signed-rank,300,4.195973e-04,8.381216e-04,***



PER-CLASS BREAKDOWN


,setting,class,support,precision,recall,f1
0,No retrieval,SUPPORT,124,0.551,0.435,0.486
1,No retrieval,CONTRADICT,64,0.301,0.672,0.415
2,No retrieval,NEI,112,0.373,0.196,0.257
3,No retrieval (matched prompt),SUPPORT,124,0.511,0.556,0.533
4,No retrieval (matched prompt),CONTRADICT,64,0.294,0.234,0.261
5,No retrieval (matched prompt),NEI,112,0.342,0.348,0.345
6,Retrieval (hybrid),SUPPORT,124,0.738,0.500,0.596
7,Retrieval (hybrid),CONTRADICT,64,0.343,0.531,0.417
8,Retrieval (hybrid),NEI,112,0.581,0.607,0.594


... (3 outputs left out)

,metric,method,score
0,Recall@5,bm25,"0.727 [0.679, 0.774]"
1,Recall@5,dense,"0.752 [0.703, 0.800]"
2,Recall@5,hybrid,"0.760 [0.714, 0.806]"
3,MRR,bm25,"0.613 [0.563, 0.660]"
4,MRR,dense,"0.638 [0.588, 0.684]"
5,MRR,hybrid,"0.671 [0.624, 0.718]"


,metric,comparison,delta,95% CI,p,p_holm,sig_holm
0,Recall@5,dense - bm25,+0.025,"[-0.019, +0.071]",0.2720,0.6768,ns
1,Recall@5,hybrid - bm25,+0.034,"[+0.007, +0.063]",0.0120,0.0600,ns
2,Recall@5,hybrid - dense,+0.009,"[-0.028, +0.044]",0.6576,0.6768,ns
3,MRR,dense - bm25,+0.026,"[-0.016, +0.067]",0.2256,0.6768,ns
4,MRR,hybrid - bm25,+0.059,"[+0.032, +0.085]",0.0002,0.0012,**
5,MRR,hybrid - dense,+0.033,"[+0.004, +0.064]",0.0212,0.0848,ns



INTERPRETATION
Accuracy      0.397 -> 0.547  (McNemar p=3.3e-05); 80 claims fixed by retrieval, 35 broken.
Hallucination 0.647 -> 0.458  (delta -0.188, 95% CI [-0.250, -0.127])
Largest per-class F1 gain from retrieval: NEI (+0.337).

Prompt-matched baseline (A2) - does the effect survive?


,metric,delta,p_holm,sig_holm
4,accuracy,+0.137,0.000499,***
5,macro_F1,+0.156,0.000600,***
6,citation (gold),+0.128,0.000838,***
7,hallucination (gold),-0.128,0.000838,***


**Threshold sensitivity**

In [16]:
rec_a = list(read_jsonl(OUT/"sentscores_noretrieval_gold.jsonl"))
rec_b = list(read_jsonl(OUT/f"sentscores_{RETRIEVER_FOR_LLM}_gold.jsonl"))
ids = [r["claim_id"] for r in pred_ret]
rows = []
for mode in ["hybrid", "lex", "nli"]:
    for lt in LEX_GRID:
        for et in ENT_GRID:
            if mode == "lex" and et != ENT_GRID[0]: continue
            if mode == "nli" and lt != LEX_GRID[0]: continue
            A = per_claim_faith(rec_a, ids, lt, et, mode); B = per_claim_faith(rec_b, ids, lt, et, mode)
            ca = np.array([A[i][1] for i in ids]); cb = np.array([B[i][1] for i in ids])
            m = ~(np.isnan(ca) | np.isnan(cb))
            rows.append({"mode": mode, "lex_t": lt if mode != "nli" else None, "ent_t": et if mode != "lex" else None,
                         "cit_A": ca[m].mean(), "cit_B": cb[m].mean(), "delta": (cb[m] - ca[m]).mean(),
                         "wilcoxon_p": wilcoxon(cb[m], ca[m], zero_method="zsplit").pvalue})
sweep = pd.DataFrame(rows); sweep.to_csv(OUT/"threshold_sensitivity.csv", index=False); display(sweep.round(3))

,mode,lex_t,ent_t,cit_A,cit_B,delta,wilcoxon_p
0,hybrid,0.2,0.5,0.408,0.610,0.202,0.000
1,hybrid,0.2,0.7,0.338,0.580,0.242,0.000
2,hybrid,0.2,0.9,0.250,0.493,0.243,0.000
3,hybrid,0.3,0.5,0.353,0.542,0.188,0.000
4,hybrid,0.3,0.7,0.274,0.498,0.224,0.000
5,hybrid,0.3,0.9,0.177,0.391,0.214,0.000
6,hybrid,0.4,0.5,0.350,0.513,0.163,0.000
7,hybrid,0.4,0.7,0.267,0.467,0.199,0.000
8,hybrid,0.4,0.9,0.171,0.343,0.172,0.000
9,hybrid,0.5,0.5,0.350,0.488,0.138,0.000


**Human Validation**

In [17]:
# rng = np.random.default_rng(SEED)
# rec = {"no_retrieval": list(read_jsonl(OUT/"sentscores_noretrieval_gold.jsonl")),
#        "retrieval":    list(read_jsonl(OUT/f"sentscores_{RETRIEVER_FOR_LLM}_gold.jsonl"))}
# by_label = {l: [e["claim_id"] for e in eval_examples if e["label"] == l] for l in LABELS}
# sample_ids = sum([list(rng.choice(v, size=17, replace=False)) for v in by_label.values()], [])  # ~51 claims
# sample = []
# for setting, rr_ in rec.items():
#     for x in rr_:
#         if x["claim_id"] in sample_ids:
#             ex = next(e for e in eval_examples if e["claim_id"] == x["claim_id"])
#             ref = " ".join(gold_ev_sents(ex["gold_doc_ids"]))[:3000]
#             sample.append({"setting": setting, **{k: x[k] for k in ("claim_id", "sentence", "max_jac", "max_ent")},
#                            "claim": ex["claim"], "gold_label": ex["label"], "reference_abstract": ref,
#                            "annotator1": "", "annotator2": ""})   # 1=supported by reference, 0=not
# pd.DataFrame(sample).sample(frac=1, random_state=SEED).to_csv(OUT/"human_eval_sheet.csv", index=False)

In [18]:
# from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support
# h = pd.read_csv(OUT/"human_eval_annotated.csv").dropna(subset=["annotator1", "annotator2"])
# print("inter-annotator kappa:", cohen_kappa_score(h.annotator1, h.annotator2))
# h["human"] = (h.annotator1 + h.annotator2 >= 1.5).astype(int)   # resolve disagreements separately
# h["auto"] = ((h.max_jac >= LEXICAL_THRESH) | (h.max_ent >= FAITH_ENTAIL_THRESH)).astype(int)
# p, r, f, _ = precision_recall_fscore_support(h.human, h.auto, average="binary")
# print(f"auto vs human: P={p:.2f} R={r:.2f} F1={f:.2f} kappa={cohen_kappa_score(h.human, h.auto):.2f}")
# print(h.groupby("setting").apply(lambda d: cohen_kappa_score(d.human, d.auto)))   # is the metric biased for one setting?

In [19]:
# Second verifier
rows = []
for d in sorted(_BASE.iterdir()):
    f = d/"faithfulness_hybrid_gold.json"; g = d/"faithfulness_noretrieval_gold.json"
    if f.exists() and g.exists():
        a, b = json.load(open(g)), json.load(open(f))
        rows.append({"verifier": d.name, "acc_A": a["accuracy"], "acc_B": b["accuracy"],
                     "cit_A": a["citation_rate"], "cit_B": b["citation_rate"]})
pd.DataFrame(rows).to_csv(_BASE/"multi_model_summary.csv", index=False)